# 04 · Export
## Producing publication-ready AFM figures

Notebooks 02 and 03 introduced how to handle AFM data within Python; how to take a NumPy array and level, mask and measure it.
This notebook is about presenting the data well: choosing a colour scale that doesn't distort your data, smoothing for display without contaminating your analysis, annotating clearly, and exporting in a format that actually serves your next step; a journal, a poster, or handing the data on to a colleague or another piece of software.

> NOTE: This notebook focuses on plotting images and doesn't deal with basic `matplotlib` plotting of lines or other plot types. There are many tutorials and guides to this already. You can start here: https://matplotlib.org/stable/tutorials/pyplot.html

**Contents**
1. Imports and data
2. Choosing a colour scale
3. Smoothing for display (not for analysis)
4. Annotating: scale bars and labels
5. Figure size, DPI, and resolution
6. Choosing an export format
7. Forwarding real data, not just pictures
8. Pulling it together: a multi-panel figure

---

## 1 · Imports and data

We are going to reuse a number of the tool we used in the previous notebooks; the colormap registration, levelling and masking functions and the `show()` plotting function.

A new sample image of a polymer thin film will be used to demonstrate the effect of different colormaps on data and how to create multi-panel figure, and we will also re-load the annexin video frame from the processing notebook. 

In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from scipy.ndimage import gaussian_filter
from skimage.color import rgb2lab
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

import tifffile

from AFMReader.general_loader import LoadFile

DATA_DIR    = Path(".") / "data" / "samples"
ARRAY_DIR   = Path(".") / "data" / "arrays"
RESULTS_DIR = Path(".") / "data" / "results"
FIG_DIR     = Path(".") / "data" / "figures"

def register_afm_colormaps(folder=Path(".") / "resources" / "colormaps"):
    """Load the workshop's AFM colormaps from CSV and register them with matplotlib.
    Safe to call more than once — already-registered colormaps are skipped."""
    for name in ("afm_brown", "playnano_gold", "classic_afm", "aurea"):
        if name in plt.colormaps():
            continue
        rgb_data = np.loadtxt(folder / f"{name}.csv", delimiter=",")
        cmap = ListedColormap(rgb_data, name=name)
        plt.colormaps.register(cmap)
        plt.colormaps.register(cmap.reversed(), name=f"{name}_r")

register_afm_colormaps()
print("Imports and colormaps ready.")


In [ ]:

# --- Tools from notebook 02 ---
def fit_plane(data, mask=None):
    """Return the best-fit plane. Fit background only if mask given (True=feature)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    model = LinearRegression().fit(coords[keep], z[keep])
    return model.predict(coords).reshape(h, w)
    
def fit_polynomial(data, order=2, mask=None):
    """Return best-fit 2D polynomial surface (background only if masked)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    A = PolynomialFeatures(order).fit_transform(coords)  # 1, x, y, x², xy, y², ...
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    coeff, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
    return (A @ coeff).reshape(h, w)
    
def row_median_align(data, mask=None):
    """Subtract each row's median (background pixels only, if mask given)."""
    out = data.copy()
    for i in range(data.shape[0]):
        ref = data[i, :] if mask is None else data[i, ~mask[i, :]]
        if ref.size == 0:
            ref = data[i, :]
        out[i, :] -= np.median(ref)
    return out

def threshold_mask(data, k=1.0):
    """Boolean mask: True where height > mean + k*std (True = feature)."""
    return data > (data.mean() + k * data.std())

print("Tools ready.")


In [ ]:

# Load and level the polymer thin film data
img_film, px_film = LoadFile(DATA_DIR / "240123_s4-023C-large.0_00050.spm",
                            channel="Height Sensor").load()
img_film = img_film.astype(np.float64)

rough_film = img_film - fit_plane(img_film)   # rough level with a plane fit
mask_rough = threshold_mask(rough_film, k=0)   # mask with a threshold
levelled_film = rough_film - fit_plane(rough_film, mask=mask_rough)   # fit a plane from the masked data
mask_film = threshold_mask(levelled_film, k=-0.1)   # re-mask the data
levelled_film = levelled_film - fit_polynomial(levelled_film, order=2, mask=mask_film)   # subtract a polynomial fit form the data

# Reload annexin data - also do a quick hole mask and plane fit.
d = np.load(ARRAY_DIR / "levelled_annexin.npz")
levelled_annexin = d["image"]
px_annexin = float(d["pixel_to_nm"])

mask_annexin = np.invert(threshold_mask(levelled_annexin, k=-1.0))
levelled_annexin = levelled_annexin - fit_plane(levelled_annexin, mask=mask_annexin)


def show(img, title, cmap="playnano_gold", zmin=None, zmax=None, ax=None, colorbar=True):
    ax = ax or plt.gca()
    im = ax.imshow(img, cmap=cmap, vmin=zmin, vmax=zmax, origin="lower")
    ax.set_title(title); ax.axis("off")
    if colorbar:
        plt.colorbar(im, ax=ax, label="Height (nm)", fraction=0.046)
    return im

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
show(levelled_film, "Levelled thin film", zmin=-10.0, zmax=30, ax=ax[0])
show(levelled_annexin, "Levelled annexin frame", zmin=-1.0, zmax=1, ax=ax[1])
plt.tight_layout()
plt.show()


## 2 · Choosing a colour scale

AFM imaging data is typically rendered as false-colour images using a range of orange-brown colour scales. The colormap you pick isn't just an aesthetic choice; a badly chosen one can make smooth data look stepped or hide real features in a flat-looking band.

`matplotlib` supplies a range of [built in colormaps](https://matplotlib.org/stable/gallery/color/colormap_reference.html) that includes 'afmhot', a common colormap for displaying AFM data that moves from black, through red/orange to yellow, and finally white. The 'classic_afm' colorscale, registered above, seeks to replicate the black-brown-white colormap supplied with the software from several AFM manufacturers.

While very commonly used, these colormaps, along with many other built-in options, can create visual artifacts in AFM data and hide details. Background features or contaminants can be hidden within wide black plateaus (see `classic_afm`), or discontinuities in the colour gradient can create the false impression of features, ledges, or stripes where there are none. This can be seen by applying various colormaps to pyramids, as is done in the next cell. Since the pyramids are smooth, your eye should only perceive a cross formed by the edges of the pyramid alongside a smooth colour gradient. Any square "banding" indicates that a discontinuity may create a false feature in an image; this can be clearly seen in the colormaps in the first row.

The second row features better options for visualising AFM data. While `viridis` is a popular perceptually uniform colormap built into matplotlib, `aurea`, `afm_brown`, and `playnano_gold` are colormaps I have engineered specifically for AFM data visualisation to resolve fine topographic features without distorting quantitative height data.

- **`afm_brown`** is an attempt to create a colormap that looks like `classic_afm` but without its disadvantages. It uses
  *monotonically increasing luminance*; brightness increases steadily with height, with no jumps. This resolves fine substrate
  detail and artificial features, while keeping the familiar orange-brown AFM look.
- **`playnano_gold`** is also monotone-lightness, spanning the *full* luminance range (black to white) for maximum contrast on
  complex, feature-rich samples.
- **`aurea`** is a gold-toned, perceptually uniform colormap designed entirely in 3D CIELAB space. It has a flat perceptual step size (ΔE≈0.45) and a
  monotonic lightness profile. The colormap preserves fine substrate detail at low heights, prevents mid-scale flattening, and guarantees accurate
  contrast for quantitative AFM analysis with a metallic gold appearance.

In [ ]:

h = w = 200
Y, X = np.mgrid[0:h, 0:w]

# Pyramid parameters
cx, cy = 100, 100   # center
height = 10         # peak value
scale = 40          # controls width of pyramid

# Pyramid surface (Manhattan distance gives a square pyramid)
pyramid = height - (np.abs(X - cx) + np.abs(Y - cy)) / scale
pyramid = np.clip(pyramid, 0, None)  # no negative values

cmaps = [
    "jet", "gist_heat","classic_afm",  "afmhot", 
    "viridis", "aurea", "afm_brown", "playnano_gold"
]

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for ax, cmap_name in zip(axes.ravel(), cmaps):
    ax.imshow(pyramid, cmap=cmap_name, origin="lower")
    ax.set_title(cmap_name)
    ax.axis("off")

plt.tight_layout()
plt.show()


When applied to the smooth test pyramids, `jet` and `afmhot` display distinct concentric squares. These look like terraces or steps in the topography, but they are entirely artificial false features caused by abrupt, non-smooth jumps in the colormap's underlying lightness profile. Similarly, `classic_afm` suffers these squares and also from a wide, flat "dead zone" at its dark end, leaving the bottom ~30% of the sample height completely black and unresolved.

The second row fixes these issues by focusing on how the human eye interprets depth. `afm_brown` intentionally recovers that lost 30% background detail; while it trades off a small amount of mild visual banding to stay within monitor limits, it offers drastically better clarity than `classic_afm` while retaining the same classic AFM look. `playnano_gold` stretches to pure white for maximum contrast, while `aurea` achieves strict mathematical uniformity (*ΔE≈0.45*) to ensure that equal changes in data height always match equal changes in perceived brightness. With these engineered scales, any edge or gradient you see in an AFM image is guaranteed to be a real, physical surface feature rather than an artifact of the color palette.

In [ ]:
for cmap_name in cmaps:
    cmap = plt.get_cmap(cmap_name)
    norm_vals = np.linspace(0, 1, 256)
    rgb = cmap(norm_vals)[:, :3].reshape(1, 256, 3)
    L = rgb2lab(rgb)[0][:, 0]
    reversals = np.sum(np.diff(np.sign(np.diff(L))) != 0)
    print(f"{cmap_name:15s} L* range [{L.min():5.1f}, {L.max():5.1f}]   sign reversals: {reversals:3d}")

This quantitative analysis confirms exactly why the top row creates visual artifacts. While `gist_heat` and `afmhot` climb monotonically, `jet` loops back and forth (5 reversals), explaining its false concentric terracing. Most strikingly, `classic_afm` suffers from 27 rapid sign reversals.

In contrast, our engineered palettes, `aurea`, `afm_brown`, and `playnano_gold`, achieve the core design goal of 0 sign reversals. Their lightness slopes climb strictly in a single direction, meaning brightness strictly tracks height. To see how these mathematical differences translate to actual scientific analysis, we can apply the same colormaps to a real-world AFM dataset instead of an idealized test geometry:

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 9))

cmaps = [
    "jet", "gist_heat","classic_afm",  "afmhot", 
    "viridis", "aurea", "afm_brown", "playnano_gold"
]

for ax, cmap_name in zip(axes.ravel(), cmaps):
    show(levelled_film, cmap_name, cmap=cmap_name, ax=ax, colorbar=False, zmin=-10.0, zmax=25.0)

plt.tight_layout()
plt.show()

## 3 · Smoothing for display (not for analysis)

A light Gaussian blur can make a figure look cleaner for a paper or poster by softening high-frequency, pixel-to-pixel sensor noise. **This is a display choice, not a processing step**, although many analysis steps are improved or rely on filtering, this is not the aim here.

`scipy.ndimage.gaussian_filter` takes `sigma` in **pixels**, not nanometres (the same blur radius means something different at different pixel sizes) so don't reuse a `sigma` value across images with different `pixel_to_nm` without checking.

In [ ]:

sigma = 1.5
smoothed = gaussian_filter(levelled_annexin, sigma=sigma)

row = levelled_annexin.shape[0] // 2
x_nm = np.arange(levelled_annexin.shape[1]) * px_annexin

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(levelled_annexin, "Raw (levelled)", ax=ax[0], zmin=-1.0, zmax= 1.0)
show(smoothed, f"Smoothed for display (sigma={sigma}px)", ax=ax[1], zmin=-1.0, zmax= 1.0)
ax[0].axhline(row, color="r", lw=1, ls="--")
ax[1].axhline(row, color="r", lw=1, ls="--")
ax[2].plot(x_nm, levelled_annexin[row, :], alpha=0.5, label="raw")
ax[2].plot(x_nm, smoothed[row, :], label="smoothed")
ax[2].legend(); ax[2].set_xlabel("Distance (nm)"); ax[2].set_ylabel("Height (nm)")
ax[2].set_title("Row profile")
plt.tight_layout()
plt.show()


`skimage.filters.gaussian` does the same job if you're already working in the `scikit-image` ecosystem, the two aren't numerically identical (they handle image edges slightly differently by default), but both give the same kind of noise reduction.

## 4 · Annotating: scale bars and labels

### Building a scale bar

A scale bar is just a rectangle of known physical length plus a text label. We build it as a reusable function with one nice trick: rather than hardcoding a length and re-tuning it for every image, we round to the nearest "clean" number, 1, 2, or 5 times a power of ten (the same rule plot-axis tick marks use) — so the bar always reads as a sensible round value like "100 nm" rather than an arbitrary "87.3 nm".

In [ ]:

def nice_round_length(target):
    """Round target to the nearest 1, 2, or 5 x a power of ten."""
    if target <= 0:
        return 1
    exponent = np.floor(np.log10(target))
    fraction = target / 10**exponent
    if fraction < np.sqrt(2):
        nice = 1
    elif fraction < np.sqrt(10):
        nice = 2
    elif fraction < np.sqrt(50):
        nice = 5
    else:
        nice = 10
    return nice * 10**exponent

def draw_scalebar(ax, pixel_to_nm, image_width_px, fraction=0.2, color="white",
                  margin_frac=0.04, text=True, units="nm"):
    """Draw a scale bar with an auto-picked round length, spanning roughly
    `fraction` of the image width. `units` controls the displayed label only
    ("nm" or "um") — the underlying geometry is unchanged, since a clean length
    in nm is always a clean length in um too (1000 being a power of ten)."""
    target_nm = fraction * image_width_px * pixel_to_nm
    bar_nm = nice_round_length(target_nm)
    bar_px = bar_nm / pixel_to_nm

    bar_value, unit_label = (bar_nm / 1000, "µm") if units == "um" else (bar_nm, "nm")

    margin = margin_frac * image_width_px
    x0 = image_width_px - margin - bar_px
    y0 = margin
    height_px = 0.015 * image_width_px

    ax.add_patch(plt.Rectangle((x0, y0), bar_px, height_px, color=color))
    if text:
        ax.text(x0 + bar_px / 2, y0 + height_px * 3, f"{bar_value:g} {unit_label}",
               color=color, ha="center", fontsize=11)

fig, ax = plt.subplots(figsize=(5, 5))
show(smoothed, "Auto-length scale bar", ax=ax, colorbar=False, zmin=-1.0, zmax=1.0)
draw_scalebar(ax, pixel_to_nm=px_annexin, image_width_px=levelled_annexin.shape[1], text=True)
plt.show()


### Other annotations

Text annotations are useful for calling out a measured value directly on the figure —
e.g. dropping in the Sq value from notebook 03 — and a bold letter in the corner is the
standard way to label panels (a, b, c...) in a multi-panel figure.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
show(levelled_annexin, "Annotated figure", ax=ax, colorbar=False, zmin=-1.0, zmax=1.0)
draw_scalebar(ax, pixel_to_nm=px_annexin, image_width_px=levelled_annexin.shape[1])

sq_value = np.sqrt(np.mean((levelled_annexin - levelled_annexin.mean())**2))
ax.text(0.03, 0.97, f"Sq = {sq_value:.2f} nm", transform=ax.transAxes,
       color="white", fontsize=11, va="top")
ax.text(0.03, 0.03, "a", transform=ax.transAxes, color="white",
       fontsize=16, fontweight="bold", va="bottom")
plt.show()

## 5 · Figure size, DPI, and resolution

The final pixel dimensions of a saved raster figure (.png, .jpg) are calculated simply as figsize (inches) × dpi. This relationship matters because academic journals specify a required DPI scaled to a target physical column layout. For poster printing or publications, you should always target a minimum of 300 DPI.

Always check your target journal's specific author guidelines, but these are standard baseline scales:

In [ ]:
for figsize, dpi in [((5, 5), 100), ((5, 5), 300), ((3.5, 3.5), 300), ((7, 5), 300)]:
    px_w, px_h = figsize[0] * dpi, figsize[1] * dpi
    print(f"figsize={figsize} in, dpi={dpi:4d}  ->  {px_w:.0f} x {px_h:.0f} px output")

print()
print("Rough common guidance (always check your target journal):")
print("  ~300 dpi at single-column width (~3.5 in) or double-column width (~7 in)")

## 6 · Choosing an export format

The choice of file format to export to depends on the quality you need and where you want to use the image. File format also determines the size of the files. 

Here, build one annotated figure and export it several ways, to compare real file sizes rather than guessing.

In [ ]:

fig, ax = plt.subplots(figsize=(5, 5))
im = show(levelled_annexin, "", ax=ax)
draw_scalebar(ax, pixel_to_nm=px_annexin, image_width_px=levelled_annexin.shape[1])
plt.tight_layout()

FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / "export_test.png", dpi=300)
plt.savefig(FIG_DIR / "export_test.jpg")
plt.savefig(FIG_DIR / "export_test.pdf")
plt.savefig(FIG_DIR / "export_test.svg")
plt.savefig(FIG_DIR / "export_test_uncompressed.tif", dpi=300)
plt.savefig(FIG_DIR / "export_test_lzw.tif", dpi=300, pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)

import os
for fname in ["export_test.png", "export_test.jpg", "export_test.pdf", "export_test.svg",
             "export_test_uncompressed.tif", "export_test_lzw.tif"]:
    size_kb = os.path.getsize(FIG_DIR / fname) / 1024
    print(f"{fname:32s}: {size_kb:8.1f} KB")
    

A few practical rules of thumb from this:

- **PNG**: simple, lossless, universally supported. A good default for screen/slides.
- **PDF / SVG** (vector): text, the scale bar, and lines stay crisp at *any* zoom or print size, and for an annotation-heavy figure like this one they
  were also considerably smaller than the high-DPI PNG. Good for journal submissions that accept vector formats.
- **TIFF**: often a journal requirement. Uncompressed TIFF is large; adding `pil_kwargs={"compression": "tiff_lzw"}` to `savefig` shrinks it losslessly
  (no quality loss) for a fraction of the size.
- **Avoid JPEG for quantitative images.** Its lossy compression can visibly distort colour values, fine for a casual photo, risky for data you're
  presenting as accurate.
- For a slide background or overlay, `plt.savefig(..., transparent=True)` drops the white background (not jpeg).

If you are working with time-series or high-speed AFM videos then I suggest you use the tools in [playNano](https://github.com/derollins/playNano) to export the data as GIFs or videos. 

## 7 · Forwarding real data, not just pictures

Everything in Section 6 exports a **picture**, RGB(A) pixel colours, matching what the colormap drew. That's right for a figure, but it's *not* your real height data anymore. If you (or a collaborator, or ImageJ, or Gwyddion) need to reopen the actual measurements, to remeasure, reprocess, or apply a different colour scale, you need to export the **array itself**, not a colourised picture of it.

`tifffile` writes a true single-channel float TIFF that preserves every real value exactly, and that ImageJ and similar tools open as quantitative data, not just an image.

In [ ]:

# Export the REAL data (float32 height array), not a colour rendering of it
tifffile.imwrite(FIG_DIR / "annexin_data.tif", levelled_annexin.astype(np.float32))

# Round-trip check: did we get the exact same numbers back?
reloaded = tifffile.imread(FIG_DIR / "annexin_data.tif")
print("Exact match after round-trip:", np.array_equal(levelled_annexin.astype(np.float32), reloaded))
print("A real pixel value:", levelled_annexin[50, 50], "nm  ->  reloaded as:", reloaded[50, 50], "nm")

# Contrast: what you get back from a colourised PNG/TIFF "picture" instead
from PIL import Image
picture = np.array(Image.open(FIG_DIR / "export_test.png"))
print("\nA pixel from the colourised picture instead:", picture[100, 100],
     "<- RGBA colour values, NOT a height in nm")


**Two things worth knowing:**

We deliberately saved the data TIFF *uncompressed*. `tifffile` can write LZW-compressed TIFFs too, but reading an LZW-compressed TIFF back with `tifffile` needs an extra package (`imagecodecs`) that isn't always installed, and ImageJ reads uncompressed TIFF natively with zero setup. For a quantitative-data export meant to travel to other software, uncompressed is the safer default; save LZW compression for the colourised *picture* exports in Section 6, which don't need to be read back into NumPy.

A bare TIFF like this preserves the real height **values**, but not necessarily the pixel size or units as embedded metadata, for that level of preservation, look into OME-TIFF (a metadata-rich TIFF variant common in microscopy) or your AFM software's own native export format.

## 8 · Pulling it together: a multi-panel figure

A small reusable helper, colourbar, optional scale bar, optional panel letter, saves repeating the same three lines on every subplot of a composite figure.

A very common, genuinely useful pattern in published AFM figures is **overview + zoom**: one panel showing the full scan for context, a second showing a cropped region at higher effective magnification, with a box on the overview marking exactly where the zoom came from. Without that box, a zoomed panel floating on its own is hard to place, the reader has no idea what part of the sample, or how much of it, they're looking at.

In [ ]:
def annotate_panel(ax, im, image_width_px, panel_label=None, scalebar_dx=None,
                   cbar_label="Height (nm)", scalebar_units="nm", text=True):
    """Add a colourbar, optional scale bar, and optional bold panel letter to an Axes."""
    plt.colorbar(im, ax=ax, label=cbar_label, fraction=0.046)
        
    if scalebar_dx is not None:
        draw_scalebar(ax, pixel_to_nm=scalebar_dx, image_width_px=image_width_px, text=text,
                     units=scalebar_units)
    if panel_label is not None:
        ax.text(0.03, 0.97, panel_label, transform=ax.transAxes, color="white",
               fontsize=16, fontweight="bold", va="top")

In [ ]:
# Zoom regions - one entry per zoomed region, try moving these about by chnaging the ranges.
zoom_regions = [
    {"y_range": (55, 225),  "x_range": (235, 415), "label": "b"},
    {"y_range": (215, 370), "x_range": (130, 360), "label": "c"},
]

# Share colour scale? 
# True compares absolute heights fairly across every panel;
# False lets each zoom use its own scale to reveal local detail 
share_colour_scale = False

# Change the colormap
colormap = "afm_brown"

# Choose the z scale, None for auto.
vmin_overview, vmax_overview = -5.0, 25.0

n_panels = 1 + len(zoom_regions)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))

# --- Panel a: the full overview, with one box per zoomed region ---
im_overview = axes[0].imshow(levelled_film, cmap=colormap,
                             vmin=vmin_overview, vmax=vmax_overview, origin="lower")
axes[0].axis("off")
axes[0].set_title("Thin film (overview)")
annotate_panel(axes[0], im_overview, levelled_film.shape[1],
              panel_label="a", scalebar_dx=px_film, scalebar_units="um")

for region in zoom_regions:
    y0, y1 = region["y_range"]
    x0, x1 = region["x_range"]
    axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                    edgecolor="white", facecolor="none", lw=1.5))

# --- One panel per zoomed region ---
for ax, region in zip(axes[1:], zoom_regions):
    y0, y1 = region["y_range"]
    x0, x1 = region["x_range"]
    crop = levelled_film[y0:y1, x0:x1]

    if share_colour_scale:
        vmin, vmax = vmin_overview, vmax_overview
    else:
        vmin, vmax = crop.min(), crop.max()

    im_crop = ax.imshow(crop, cmap=colormap, vmin=vmin, vmax=vmax, origin="lower")
    ax.axis("off")
    ax.set_title(f"Zoomed region ({region['label']})")
    annotate_panel(ax, im_crop, crop.shape[1], panel_label=region["label"], scalebar_dx=px_film, scalebar_units="um")

plt.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / "composite_figure.pdf")
plt.show()
print("Saved:", FIG_DIR / "composite_figure.pdf")

---

### Recap

- Pick a colormap with **monotonic lightness** (`aurea`, `afm_brown`, `playnano_gold` or `viridis`) unless you have a specific reason (legacy continuity)
  to accept `classic_afm`'s non-linearity,  and steer clear of rainbow scales like `jet` for quantitative data.
- Build a scale bar with `draw_scalebar`, auto-picking a clean round length (1/2/5 × a power of ten) means you never have to manually tune it per image.
- **Pixels = figsize × dpi.** Vector formats (PDF/SVG) keep text and lines crisp at any size; TIFF needs LZW compression to avoid huge files;   avoid JPEG for publication.
- A colourised export is a **picture**, not your data. If something downstream needs to reanalyse the real values, export the array itself (`tifffile`),  not a screenshot of a colourful version of it.

That's the full pipeline: load (01) → level and mask (02) → measure (03) → present (04).